## Practical 1

In [ ]:
from pathlib import Path
import warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
ROOT = Path.cwd()
POLL = ['PM2.5','PM10','NO2','SO2','CO','O3']
print('Notebook working folder:',ROOT)
print('CSV files:',[p.name for p in ROOT.glob('*.csv')])

In [ ]:
# Optional public API demonstration (internet required); not used as training data.
# Open-Meteo air quality API: Nagpur coordinates, past/current hourly estimates.
import requests
try:
    url='https://air-quality-api.open-meteo.com/v1/air-quality'
    params={'latitude':21.1458,'longitude':79.0882,'hourly':'pm10,pm2_5,nitrogen_dioxide,ozone','forecast_days':1,'timezone':'Asia/Kolkata'}
    response=requests.get(url,params=params,timeout=12); response.raise_for_status()
    api_data=response.json().get('hourly',{})
    api_df=pd.DataFrame(api_data)
    print('Open-Meteo API sample (separate from training):'); display(api_df.head())
except Exception as exc:
    print('API unavailable (offline is OK):',str(exc)[:160])

In [ ]:
# Load the actual dataset from the VS Code folder.
path = ROOT / 'air_quality_cleaned.csv'
if not path.exists():
    raise FileNotFoundError(f'Put air_quality_cleaned.csv next to this notebook in {ROOT}.')
raw = pd.read_csv(path)
print('Original data:',raw.shape); display(raw.head()); raw.info()
print('Missing values:'); display(raw.isna().sum().sort_values(ascending=False))
print('Duplicate rows:',raw.duplicated().sum())
required=['AQI'] + POLL
missing=[col for col in required if col not in raw.columns]
if missing: raise ValueError(f'Missing required columns: {missing}; present: {raw.columns.tolist()}')

In [ ]:
df=raw.drop_duplicates().copy()
for col in required: df[col]=pd.to_numeric(df[col],errors='coerce')
if 'Date' in df:
    df['Date']=pd.to_datetime(df['Date'],errors='coerce')
    df['Month']=df['Date'].dt.month
if 'Month' not in df: df['Month']=6  # fallback only when the dataset lacks dates
season_map={12:'Winter',1:'Winter',2:'Winter',3:'Summer',4:'Summer',5:'Summer',6:'Monsoon',7:'Monsoon',8:'Monsoon',9:'Monsoon',10:'Post-Monsoon',11:'Post-Monsoon'}
if 'Season' not in df: df['Season']=df['Month'].map(season_map)
df['Season']=df['Season'].fillna('Unknown').astype(str)
if 'City' not in df: df['City']='Unknown'
df['City']=df['City'].fillna('Unknown').astype(str)
df=df.dropna(subset=['AQI']).copy()
# For exploratory analysis only; model pipelines impute using training split to avoid leakage.
for col in POLL: df[col]=df[col].fillna(df[col].median())
if df[POLL].isna().any().any(): raise ValueError('One or more pollutants have no usable numeric values.')
def aqi_bucket(aqi):
    if aqi <=50:return 'Good'
    if aqi <=100:return 'Satisfactory'
    if aqi <=200:return 'Moderate'
    if aqi <=300:return 'Poor'
    if aqi <=400:return 'Very Poor'
    return 'Severe'
df['AQI_Bucket']=df['AQI'].map(aqi_bucket)
print('Analysis data:',df.shape); display(df.head())
print('AQI classes:'); display(df['AQI_Bucket'].value_counts())

## Practical 2

In [ ]:
display(df.describe(include='all').T)
print('Highest AQI observations:'); display(df.nlargest(5,'AQI')[['City','AQI','AQI_Bucket']])
print('Highly polluted readings:',(df['AQI']>200).sum())
print('Mean AQI by City:'); display(df.groupby('City')['AQI'].agg(['count','mean','median']).sort_values('mean',ascending=False).head(10))
print('City × Season pivot table:'); display(pd.pivot_table(df,index='City',columns='Season',values='AQI',aggfunc='mean').head(10))
q1,q3=df['AQI'].quantile([.25,.75]); iqr=q3-q1
print('AQI IQR limits:',round(q1-1.5*iqr,2),round(q3+1.5*iqr,2),'outliers:',int(((df['AQI']<q1-1.5*iqr)|(df['AQI']>q3+1.5*iqr)).sum()))

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(16,9))
sns.histplot(df['AQI'],bins=35,ax=axes[0,0]);axes[0,0].set_title('AQI histogram')
sns.boxplot(x=df['AQI'],ax=axes[0,1]);axes[0,1].set_title('AQI boxplot')
df.groupby('City')['AQI'].mean().nlargest(10).plot(kind='bar',ax=axes[0,2],title='Mean AQI by city');axes[0,2].tick_params(axis='x',rotation=60)
sns.scatterplot(data=df.sample(min(2000,len(df)),random_state=42),x='PM2.5',y='AQI',s=12,ax=axes[1,0]);axes[1,0].set_title('Scatter plot')
df['AQI_Bucket'].value_counts().plot.pie(autopct='%1.1f%%',ax=axes[1,1]);axes[1,1].set_ylabel('');axes[1,1].set_title('AQI categories')
sns.heatmap(df[POLL+['AQI']].corr(),annot=True,fmt='.2f',ax=axes[1,2],cmap='coolwarm');axes[1,2].set_title('Correlation heatmap')
plt.tight_layout();plt.show()

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4))
sns.violinplot(data=df,x='Season',y='AQI',ax=ax[0]);ax[0].tick_params(axis='x',rotation=30)
sns.countplot(data=df,x='AQI_Bucket',order=df['AQI_Bucket'].value_counts().index,ax=ax[1]);ax[1].tick_params(axis='x',rotation=45)
sns.regplot(data=df.sample(min(2500,len(df)),random_state=42),x='PM10',y='AQI',scatter_kws={'s':7},ax=ax[2]);
plt.tight_layout();plt.show()

## Practical 3

In [ ]:
fig,ax=plt.subplots(1,4,figsize=(17,3.5))
ax[0].hist(rng.uniform(df.AQI.min(),df.AQI.max(),2000),bins=30);ax[0].set_title('Uniform')
ax[1].hist(rng.normal(df.AQI.mean(),df.AQI.std(),2000),bins=30);ax[1].set_title('Normal')
p=(df.AQI>200).mean()
ax[2].hist(rng.binomial(10,p,2000),bins=np.arange(12)-.5);ax[2].set_title('Binomial: polluted days')
ax[3].hist(rng.poisson(max(p*10,0.01),2000),bins=20);ax[3].set_title('Poisson example')
plt.tight_layout();plt.show()
print('P(AQI>200)=',round(p,4),'P(AQI<=200)=',round(1-p,4))
print('P(3 high-AQI readings in 10 independent trials)=',round(stats.binom.pmf(3,10,p),5))

In [ ]:
n=min(150,len(df)); k=max(1,n//10)
sampling={
 'Simple random':df.sample(n=n,random_state=42),
 'With replacement':df.sample(n=n,replace=True,random_state=42),
 'Without replacement':df.sample(n=n,replace=False,random_state=42),
 'Fractional':df.sample(frac=min(.1,1),random_state=42),
 'Systematic':df.iloc[::max(1,len(df)//n)].head(n),
 'Stratified (AQI category)':df.groupby('AQI_Bucket',group_keys=False).sample(frac=min(.1,1),random_state=42),
 'Cluster (city)':df[df.City.isin(rng.choice(df.City.unique(),size=min(2,df.City.nunique()),replace=False))]
}
display(pd.DataFrame([{'Sampling':key,'Size':len(sample),'Mean AQI':sample.AQI.mean()} for key,sample in sampling.items()]))
print('Empirical probability by AQI bucket:');display(df.AQI_Bucket.value_counts(normalize=True).round(3))

## Practical 4

In [ ]:
x=df.AQI.to_numpy(); sample_size=min(50,len(x))
means=np.array([rng.choice(x,size=sample_size,replace=True).mean() for _ in range(1000)])
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].hist(x,bins=35);ax[0].set_title('Population AQI')
ax[1].hist(means,bins=35);ax[1].axvline(x.mean(),color='red',label='Population mean');ax[1].legend();ax[1].set_title('CLT: 1000 sample means')
plt.tight_layout();plt.show()
print('Population mean:',round(x.mean(),2),'Sample mean estimate:',round(means.mean(),2),'Standard error estimate:',round(means.std(ddof=1),2))

## Practical 5

In [ ]:
x=df.AQI.dropna().to_numpy(); n=len(x); mu=x.mean(); s=x.std(ddof=1); se=s/np.sqrt(n)
for level in [.90,.95,.99]:
    margin=stats.t.ppf((1+level)/2,n-1)*se
    print(f'{level:.0%} CI for mean AQI: ({mu-margin:.2f},{mu+margin:.2f})')
boots=np.array([rng.choice(x,size=n,replace=True).mean() for _ in range(1000)]) if n<=12000 else np.array([rng.choice(x,size=3000,replace=True).mean() for _ in range(1000)])
print('Bootstrap percentile interval:',np.round(np.quantile(boots,[.025,.975]),2))
plt.hist(boots,bins=30);plt.title('Bootstrap distribution of sample mean');plt.show()

In [ ]:
# Demonstration: two-sided hypothesis H0: population mean AQI = 100
mu0=100.0
z=(mu-mu0)/se; pz=2*stats.norm.sf(abs(z))
t,p_t=stats.ttest_1samp(x,popmean=mu0)
print(f'Z approximation: z={z:.3f}, p={pz:.5g}')
print(f'One-sample t-test: t={t:.3f}, p={p_t:.5g}')
print('Decision at 5%:', 'Reject H0' if p_t<.05 else 'Fail to reject H0')
print('Note: daily city measurements may not be independent; interpret significance cautiously.')

## Practical 6

In [ ]:
print('Covariance:');display(df[POLL+['AQI']].cov().round(2))
print('Pearson correlations:');display(df[POLL+['AQI']].corr('pearson').round(3))
rows=[]
for v in POLL:
    r,pv=stats.pearsonr(df[v],df.AQI)
    rho,sp=stats.spearmanr(df[v],df.AQI)
    rows.append([v,r,pv,rho,sp])
display(pd.DataFrame(rows,columns=['Pollutant','Pearson r','Pearson p','Spearman rho','Spearman p']).round(4))

In [ ]:
cities=df.City.value_counts().index.tolist()
if len(cities)>=2:
    a=df.loc[df.City==cities[0],'AQI']; bb=df.loc[df.City==cities[1],'AQI']
    t,p=stats.ttest_ind(a,bb,equal_var=False)
    print('Welch two-sample t test:',cities[0],'vs',cities[1],'t=',round(t,3),'p=',p)
else: print('Two-city comparison skipped: only one city')
groups=[g.AQI.to_numpy() for _,g in df.groupby('Season') if len(g)>=2]
if len(groups)>=2:
    F,p=stats.f_oneway(*groups); print('One-way ANOVA by season: F=',round(F,3),'p=',p)
contingency=pd.crosstab(df.Season,df.AQI_Bucket)
if contingency.shape[0]>1 and contingency.shape[1]>1:
    chi,p,dof,expected=stats.chi2_contingency(contingency)
    print('Chi-square (season x AQI bucket):',round(chi,3),'p=',p)
    display(contingency)
print('Warning: serial dependence and repeated city measurements can affect p-values.')

## Practical 7

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error, accuracy_score, f1_score, classification_report, confusion_matrix)
NUM=POLL; CAT=['City','Season']; FEATURES=NUM+CAT
X=df[FEATURES]; y=df['AQI']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
def preprocessor():
    num=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
    cat=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    return ColumnTransformer([('num',num,NUM),('cat',cat,CAT)])
reg_results=[]; reg_models={}; reg_predictions={}
cv=KFold(n_splits=3,shuffle=True,random_state=42)
def fit_regressor(name,estimator):
    pipe=Pipeline([('pre',preprocessor()),('model',estimator)])
    cv_rmse=-cross_val_score(pipe,X_train,y_train,cv=cv,scoring='neg_root_mean_squared_error',n_jobs=1).mean()
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    reg_results.append({'Model':name,'CV RMSE':cv_rmse,'Test RMSE':np.sqrt(mean_squared_error(y_test,pred)),'Test MAE':mean_absolute_error(y_test,pred),'Test R2':r2_score(y_test,pred)})
    reg_models[name]=pipe; reg_predictions[name]=pred
    print(name);display(pd.DataFrame([reg_results[-1]]).round(3))
print('Training rows:',len(X_train),'Testing rows:',len(X_test))

In [ ]:
# Multiple Linear Regression
fit_regressor('Multiple Linear Regression', LinearRegression())

In [ ]:
# Ridge Regression
fit_regressor('Ridge Regression', Ridge(alpha=10))

In [ ]:
# Lasso Regression
fit_regressor('Lasso Regression', Lasso(alpha=1.0,max_iter=6000))

In [ ]:
# Decision Tree Regression
fit_regressor('Decision Tree Regression', DecisionTreeRegressor(max_depth=5,random_state=42))

In [ ]:
# Random Forest Regression
fit_regressor('Random Forest Regression', RandomForestRegressor(n_estimators=100,max_depth=5,random_state=42,n_jobs=-1))

In [ ]:
# Compare all five regression models
reg_table=pd.DataFrame(reg_results).sort_values('CV RMSE').reset_index(drop=True)
display(reg_table.round(3))
best_name=reg_table.iloc[0]['Model']; best_pipe=reg_models[best_name]
print('Selected model using training cross-validation:',best_name)
fig,axs=plt.subplots(1,3,figsize=(17,4))
reg_table.plot.bar(x='Model',y='CV RMSE',ax=axs[0],legend=False,title='CV RMSE (lower better)')
reg_table.plot.bar(x='Model',y='Test RMSE',ax=axs[1],legend=False,title='Test RMSE')
reg_table.plot.bar(x='Model',y='Test R2',ax=axs[2],legend=False,title='Test R² (higher better)')
for ax in axs:ax.tick_params(axis='x',rotation=65)
plt.tight_layout();plt.show()

In [ ]:
pred_comparison=pd.DataFrame({'Actual AQI':y_test.to_numpy(),**{k:v for k,v in reg_predictions.items()}})
display(pred_comparison.head(12).round(2))

In [ ]:
pred=best_pipe.predict(X_test)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].scatter(y_test,pred,s=8,alpha=.35); ax[0].set_xlabel('Actual AQI');ax[0].set_ylabel('Predicted AQI');ax[0].set_title(best_name)
ax[1].hist(y_test-pred,bins=35);ax[1].set_title('Residuals (actual - predicted)')
plt.tight_layout();plt.show()
print('Test RMSE:',round(np.sqrt(mean_squared_error(y_test,pred)),3))

## Practical 8

In [ ]:
labels=df['AQI_Bucket']; counts=labels.value_counts()
stratify=labels if counts.min()>=2 and (counts*0.2).min()>=1 else None
Xc_train,Xc_test,yc_train,yc_test=train_test_split(X,labels,test_size=.2,random_state=42,stratify=stratify)
class_results=[];class_models={};class_predictions={}
def fit_classifier(name,estimator):
    pipe=Pipeline([('pre',preprocessor()),('model',estimator)])
    pipe.fit(Xc_train,yc_train); yp=pipe.predict(Xc_test)
    class_results.append({'Model':name,'Accuracy':accuracy_score(yc_test,yp),'Macro F1':f1_score(yc_test,yp,average='macro',zero_division=0)})
    class_models[name]=pipe;class_predictions[name]=yp
    display(pd.DataFrame([class_results[-1]]).round(3))
print('Class distribution:');display(labels.value_counts())

In [ ]:
# Logistic Regression
fit_classifier('Logistic Regression', LogisticRegression(max_iter=2000))

In [ ]:
# KNN
fit_classifier('KNN', KNeighborsClassifier(n_neighbors=5))

In [ ]:
# Naive Bayes
fit_classifier('Naive Bayes', GaussianNB())

In [ ]:
# Decision Tree Classification
fit_classifier('Decision Tree', DecisionTreeClassifier(max_depth=5,random_state=42))

In [ ]:
# Random Forest Classification
fit_classifier('Random Forest', RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42,n_jobs=-1))

In [ ]:
# SVM
fit_classifier('Support Vector Machine', SVC(kernel="rbf"))

In [ ]:
# Compare all six classification models
class_table=pd.DataFrame(class_results).sort_values('Macro F1',ascending=False).reset_index(drop=True)
display(class_table.round(3))
best_classifier_name=class_table.iloc[0]['Model'];best_classifier=class_models[best_classifier_name]
print('Selected classifier:',best_classifier_name)
fig,axs=plt.subplots(1,2,figsize=(13,4))
class_table.plot.bar(x='Model',y='Accuracy',ax=axs[0],legend=False,title='Accuracy')
class_table.plot.bar(x='Model',y='Macro F1',ax=axs[1],legend=False,title='Macro F1')
for ax in axs:ax.tick_params(axis='x',rotation=55)
plt.tight_layout();plt.show()

In [ ]:
print(classification_report(yc_test,best_classifier.predict(Xc_test),zero_division=0))
classes=best_classifier.classes_
sns.heatmap(pd.DataFrame(confusion_matrix(yc_test,best_classifier.predict(Xc_test),labels=classes),index=classes,columns=classes),annot=True,fmt='d',cmap='Blues')
plt.title('AQI category confusion matrix');plt.tight_layout();plt.show()

In [ ]:
# Export the selected regression + classification models, city/season profiles and calibrated location effects.
# Location offsets adjust for city/season patterns *not already explained* by the pollutants.
import sklearn
from collections import defaultdict

# Estimate residual city/season effects from held-out training folds, not the test split.
# Shrink small groups towards zero to reduce noisy offsets.
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
calibration_cv = KFold(n_splits=3, shuffle=True, random_state=42)
oof_predictions = cross_val_predict(clone(best_pipe), X_train, y_train, cv=calibration_cv, n_jobs=1)
calibration = X_train[['City','Season']].reset_index(drop=True).copy()
calibration['residual'] = y_train.to_numpy() - oof_predictions

def shrunk_offsets(frame, group_cols, prior=70):
    table = frame.groupby(group_cols, dropna=False)['residual'].agg(['mean','count']).reset_index()
    result = {}
    for _, row in table.iterrows():
        key = '||'.join(str(row[c]) for c in group_cols)
        result[key] = float(row['mean'] * row['count'] / (row['count'] + prior))
    return result

# Make offsets additive but avoid double counting city-season interactions.
city_offsets = shrunk_offsets(calibration, ['City'])
calibration['remainder'] = calibration['residual'] - calibration['City'].map(city_offsets)
season_offsets = shrunk_offsets(calibration.rename(columns={'remainder':'residual_base'}).drop(columns='residual').rename(columns={'residual_base':'residual'}), ['Season'])

# Typical measured pollutant values per city-season, for auto-filled app inputs.
group_profiles = {}
for (city, season), group in df.groupby(['City','Season']):
    group_profiles[f'{city}||{season}'] = {k:float(v) for k,v in group[POLL].median().items()}
city_profiles = {str(city): {k:float(v) for k,v in g[POLL].median().items()} for city,g in df.groupby('City')}
season_profiles = {str(season): {k:float(v) for k,v in g[POLL].median().items()} for season,g in df.groupby('Season')}

try:
    fnames = best_pipe.named_steps['pre'].get_feature_names_out()
    mdl = best_pipe.named_steps['model']
    values = getattr(mdl,'feature_importances_',None)
    if values is None: values = getattr(mdl,'coef_',None)
    importances = dict(zip(fnames,np.abs(np.asarray(values)).reshape(-1).astype(float))) if values is not None else {}
except Exception:
    importances = {}

bundle = {
    'pipeline':best_pipe, 'name':best_name,
    'metrics':reg_table.loc[reg_table.Model==best_name].iloc[0].to_dict(),
    'table':reg_table,
    'cities':sorted(df.City.unique().tolist()),
    'seasons':sorted(df.Season.unique().tolist()),
    'pollutant_medians':df[POLL].median().to_dict(),
    'group_profiles':group_profiles,
    'city_profiles':city_profiles,
    'season_profiles':season_profiles,
    'city_offsets':city_offsets,
    'season_offsets':season_offsets,
    'features':FEATURES,'importances':importances,
    'sklearn_version':sklearn.__version__,
    'classifier_pipeline':best_classifier,'classifier_name':best_classifier_name
}
output = ROOT/'aqi_model.pkl'
with output.open('wb') as file: pickle.dump(bundle,file)
with output.open('rb') as file: check = pickle.load(file)
print('MODEL SAVED:',output)
print('Cities:',len(city_profiles),'Seasons:',len(season_profiles))
print('File size (MB):',round(output.stat().st_size/1e6,2))

def predict_location_aqi(row, model_bundle):
    base = float(model_bundle['pipeline'].predict(row)[0])
    city = str(row.iloc[0]['City'])
    season = str(row.iloc[0]['Season'])
    adjusted = base + model_bundle['city_offsets'].get(city, 0.0) + model_bundle['season_offsets'].get(season,0.0)
    return max(0.0, adjusted)

example = pd.DataFrame([{**df[POLL].median().to_dict(), 'City':str(df.City.mode().iloc[0]), 'Season':str(df.Season.mode().iloc[0])}])
print('Sample predicted AQI:',round(predict_location_aqi(example,check),2))
print('Sample predicted category:',aqi_bucket(predict_location_aqi(example,check)))
print('Note: city/season corrections are statistical estimates, not live readings.')


In [ ]:
with (ROOT/'aqi_model.pkl').open('rb') as f: deployed=pickle.load(f)
print('Loaded model:',deployed['name'])
print('AQI:',round(predict_location_aqi(example,deployed),2))
print('Category:',aqi_bucket(predict_location_aqi(example,deployed)))
print('City/season calibration present:',bool(deployed['city_offsets']),bool(deployed['season_offsets']))

# Check changing city/season with identical pollutant inputs.
city_list=deployed['cities']; season_list=deployed['seasons']
for city in city_list[:3]:
    for season in season_list[:3]:
        test_row=example.copy();test_row['City']=city;test_row['Season']=season
        print(city,season,round(predict_location_aqi(test_row,deployed),2))
